# Chapter 6 — Molecular Identity & Standardization (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Parse SMILES/InChI/SDF robustly with RDKit
- Handle salts/fragments and choose a parent
- Detect stereochemistry and tautomer policy issues
- Produce duplicate/conflict reports

> Runtime: ~5 min (local, no API)  
> Cost: free  
> Data: small built-in molecule set

> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.



## Environment setup


### Secrets (optional LLM only)


In [1]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

✅ API keys loaded for OPENAI (source: Colab Secrets)


### Install pinned dependencies


In [2]:
# @title Installing Python dependencies
%pip install -q "rdkit" "langchain" "langchain-core" "langchain-openai" "pandas" "matplotlib" "scipy" "numpy<2" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 813.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 55.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but yo

In [3]:
# @title Setting LangSmith variables
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter6-mol-identity"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local/RDKit-first)")

LangSmith ON -> lc4lsh-chapter6-mol-identity


## Why molecular identity matters

Before any modeling, you must know **which molecule** a record refers to. The same compound appears as many SMILES strings, salt forms, and tautomers. Standardization makes identity **reproducible and comparable**.


## 1. Parse SMILES and report problems


In [4]:
from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

SMILES = ["CC(=O)OC1=CC=CC=C1C(=O)O", "c1ccccc1", "not_a_smiles", "C(C(C(C"]
for s in SMILES:
    m = Chem.MolFromSmiles(s)
    print(f"{s:35} -> {'OK ' + Chem.MolToSmiles(m) if m else 'PARSE FAILED'}")

CC(=O)OC1=CC=CC=C1C(=O)O            -> OK CC(=O)Oc1ccccc1C(=O)O
c1ccccc1                            -> OK c1ccccc1
not_a_smiles                        -> PARSE FAILED
C(C(C(C                             -> PARSE FAILED


## 2. Canonicalization & InChIKey as identity


In [5]:
from rdkit.Chem import inchi

asp1 = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
asp2 = Chem.MolFromSmiles("O=C(O)c1ccccc1OC(=O)C")
print("canonical 1:", Chem.MolToSmiles(asp1))
print("canonical 2:", Chem.MolToSmiles(asp2))
print("same canonical?", Chem.MolToSmiles(asp1) == Chem.MolToSmiles(asp2))
print("InChIKey:", inchi.MolToInchiKey(asp1))

canonical 1: CC(=O)Oc1ccccc1C(=O)O
canonical 2: CC(=O)Oc1ccccc1C(=O)O
same canonical? True
InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N


## 3. Salts & fragments: pick the parent


In [6]:
from rdkit.Chem import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

salt = Chem.MolFromSmiles("[Na+].CC(=O)[O-].c1ccncc1")
remover = SaltRemover.SaltRemover()
print("stripped:", Chem.MolToSmiles(remover.StripMol(salt)))
largest = rdMolStandardize.LargestFragmentChooser().choose(salt)
print("largest fragment:", Chem.MolToSmiles(largest))

stripped: c1ccncc1
largest fragment: c1ccncc1


## 4. Stereochemistry & tautomer policy


In [7]:
chiral = Chem.MolFromSmiles("C[C@H](O)c1ccccc1")
Chem.AssignStereochemistry(chiral, cleanIt=True, force=True)
flags = [a.GetProp("_CIPCode") for a in chiral.GetAtoms() if a.HasProp("_CIPCode")]
print("stereocenters (CIP):", flags)

te = rdMolStandardize.TautomerEnumerator()
keto = Chem.MolFromSmiles("CC(=O)C")
enol = Chem.MolFromSmiles("CC(O)=C")
print("canonical tautomer keto:", Chem.MolToSmiles(te.Canonicalize(keto)))
print("canonical tautomer enol:", Chem.MolToSmiles(te.Canonicalize(enol)))

stereocenters (CIP): ['S']
canonical tautomer keto: CC(C)=O
canonical tautomer enol: CC(C)=O


## 5. Duplicate & conflict report


In [8]:
!pip install -q --ignore-installed numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 785.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 55.5 MB/s eta 0:00:00


In [9]:
import pandas as pd

records = pd.DataFrame(
    {
        "name": ["aspirin-a", "aspirin-b", "benzene", "asp-dup", "bad"],
        "smiles": [
            "CC(=O)Oc1ccccc1C(=O)O",
            "O=C(O)c1ccccc1OC(=O)C",
            "c1ccccc1",
            "CC(=O)Oc1ccccc1C(=O)O",
            "not_valid",
        ],
    }
)


def canon(s):
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m) if m else None


records["canonical"] = records["smiles"].map(canon)
records["inchikey"] = records["canonical"].map(
    lambda c: inchi.MolToInchiKey(Chem.MolFromSmiles(c)) if c else None
)

dup = records[records.duplicated("inchikey", keep=False) & records["inchikey"].notna()]
invalid = records[records["canonical"].isna()]
print("DUPLICATES (same InChIKey):")
print(dup[["name", "inchikey"]])
print("\nINVALID (parse failed):")
print(invalid[["name", "smiles"]])

DUPLICATES (same InChIKey):
        name                     inchikey
0  aspirin-a  BSYNRYMUTXBXSQ-UHFFFAOYSA-N
1  aspirin-b  BSYNRYMUTXBXSQ-UHFFFAOYSA-N
3    asp-dup  BSYNRYMUTXBXSQ-UHFFFAOYSA-N

INVALID (parse failed):
  name     smiles
4  bad  not_valid


## Limitations & safety notes

- Standardization encodes a **policy** (salt/parent/tautomer rules); document which rules you applied.
- InChIKey collapses some stereochemical/protomer distinctions depending on options.
- Identity != activity/toxicity; this notebook only establishes *which compound* a record is.
- Local/free; no API needed.


In [10]:
# Cleanup
import gc

for _v in ("mol", "mols", "df", "llm", "model", "img", "raw", "curated"):
    globals().pop(_v, None)
try:
    import torch

    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why canonicalize SMILES?</summary>Many valid SMILES describe one molecule; a canonical form gives a single string for de-duplication and joins.</details>

<details><summary>Why use InChIKey for identity?</summary>It is a fixed-length hash of the structure, robust to representation differences and easy to index.</details>

<details><summary>Why strip salts/choose a parent?</summary>Salt forms share the active parent; collapsing them avoids counting the same drug as multiple compounds.</details>

### Tasks
- **Task A** - Add a `policy` string recording which standardization steps you applied.
- **Task B** - Detect molecules whose canonical tautomer differs from the input and flag them.
- **Task C** - Count stereoisomers that share an InChIKey-connectivity layer.
- **Task D** - Export the curated table (name, canonical, InChIKey, flags) to CSV.
